### 1. Kütüphane Yükleme
Bu blokta, veri manipülasyonu, makine öğrenimi modelleme ve performans değerlendirmesi için gerekli temel Python kütüphaneleri yükleniyor.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

### 2. Veri Yükleme ve İlk İnceleme
İki farklı CSV dosyası (googleplaystore.csv ve googleplaystore_user_reviews.csv) DataFrame'lere yükleniyor ve temel bilgiler (boyut ve ilk 5 satır) ekrana yazdırılıyor.

In [ ]:
df = pd.read_csv("DataSet/googleplaystore.csv")

reviews_df = pd.read_csv(
    "DataSet/googleplaystore_user_reviews.csv",
    encoding="latin1",        # Unicode hatasını çözer
    engine="python",          # Parser daha esnek
    on_bad_lines="skip"       # Bozuk satırları atlar
)

print("Apps Shape:", df.shape)
print("Reviews Shape:", reviews_df.shape)
print(df.head())
print(reviews_df.head())


Apps Shape: (10841, 13)
Reviews Shape: (64294, 5)
                                                 App        Category  Rating  \
0     Photo Editor & Candy Camera & Grid & ScrapBook  ART_AND_DESIGN     4.1   
1                                Coloring book moana  ART_AND_DESIGN     3.9   
2  U Launcher Lite – FREE Live Cool Themes, Hide ...  ART_AND_DESIGN     4.7   
3                              Sketch - Draw & Paint  ART_AND_DESIGN     4.5   
4              Pixel Draw - Number Art Coloring Book  ART_AND_DESIGN     4.3   

  Reviews  Size     Installs  Type Price Content Rating  \
0     159   19M      10,000+  Free     0       Everyone   
1     967   14M     500,000+  Free     0       Everyone   
2   87510  8.7M   5,000,000+  Free     0       Everyone   
3  215644   25M  50,000,000+  Free     0           Teen   
4     967  2.8M     100,000+  Free     0       Everyone   

                      Genres Last Updated         Current Ver   Android Ver  
0               Art & Design     7-J

### 3. Veri Birleştirme, Temizleme ve Hedef Değişken Oluşturma: 
- Bu blokta ana veri setleri birleştiriliyor, temizleniyor ve sınıflandırma için bir Hedef Değişken (Success) oluşturuluyor.Duygu Analizi Verisi Birleştirme: Kullanıcı yorumlarındaki ortalama Sentiment_Polarity ve Sentiment_Subjectivity değerleri hesaplanarak ana uygulama verisine (df) sol birleştirme (left merge) ile ekleniyor.Tekrar Eden Kayıtları Temizleme: Uygulamalar Last Updated (Son Güncelleme) tarihine göre sıralanıp, tekrar eden uygulama isimlerinden sadece en güncel olanı tutuluyor (drop_duplicates).Eksik Derecelendirme (Rating) Silme: Modelin hedef değişkeni Rating'e dayandığı için, Rating sütunundaki eksik değer içeren satırlar siliniyor.Hedef Değişken (Success) Oluşturma: Uygulamanın başarılı olup olmadığını belirlemek için ikili (binary) bir sınıflandırma değişkeni oluşturuluyor:Rating 4.0 ve üzeriyse = 1 (Başarılı) Aksi takdirde = 0 (Başarısız)

In [ ]:

sent_agg = reviews_df.groupby("App")[["Sentiment_Polarity", "Sentiment_Subjectivity"]].mean().reset_index()
sent_agg.columns = ["App", "Avg_Sentiment_Polarity", "Avg_Sentiment_Subjectivity"]

df = df.merge(sent_agg, on="App", how="left")


df["Last Updated"] = pd.to_datetime(df["Last Updated"], format="%b %d, %Y", errors="coerce")
df = df.sort_values("Last Updated", ascending=False).drop_duplicates(subset=["App"], keep="first")


df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
df = df.dropna(subset=["Rating"])


df["Success"] = (df["Rating"] >= 4.0).astype(int)

print("\nTarget distribution (Success):")
print(df["Success"].value_counts(normalize=True))


Target distribution (Success):
Success
1    0.766988
0    0.233012
Name: proportion, dtype: float64


Çıktı Yorumu: Hedef değişken dengesiz (imbalanced). Başarılı (1) sınıfı, verinin yaklaşık %77'sini oluşturuyor.

### 4. Özellik Mühendisliği (Feature Engineering):
- Bu fonksiyon, modelin kullanacağı özellikleri oluşturmak ve mevcut özellikleri temizleyip sayısal formata dönüştürmek için tanımlanmıştır.Temizleme İşlemleri: Reviews, Installs ve Price sütunları sayısal değere dönüştürülürken, metinsel ifadeler ('+', ',', '$') kaldırılıyor.Size Dönüşümü: Megabayt ('M') ve Kilobayt ('k') ifadeleri temizleniyor ve 'Varies with device' (Cihaza göre değişir) değerleri eksik değer (np.nan) olarak işaretleniyor.Yeni Özellikler:Review_Rate: Uygulama başına düşen yorum oranı Reviews / (Installs + 1).Recency_Days: Son güncellemeden günümüze (1 Ağustos 2018) kadar geçen süre (gün cinsinden).Price_Category: Price sütununu kategorik gruplara ayıran yeni bir özellik (ücretsiz, ucuz, düşük, orta, yüksek).

In [ ]:
def clean_and_engineer_features(df_):
    df_ = df_.copy()

    # Reviews
    df_["Reviews"] = pd.to_numeric(df_["Reviews"], errors="coerce")

    # Installs: remove '+' and ',' -> numeric
    df_["Installs"] = (
        df_["Installs"]
        .astype(str)
        .str.replace("+", "", regex=False)
        .str.replace(",", "", regex=False)
    )
    df_["Installs"] = pd.to_numeric(df_["Installs"], errors="coerce")

    # Price: remove '$'
    df_["Price"] = df_["Price"].astype(str).str.replace("$", "", regex=False)
    df_["Price"] = pd.to_numeric(df_["Price"], errors="coerce")

    # Size: remove 'M' / 'k' and handle 'Varies with device'
    size = df_["Size"].astype(str)
    size = size.str.replace("M", "", regex=False)
    size = size.str.replace("k", "", regex=False)
    size = size.replace("Varies with device", np.nan)
    df_["Size"] = pd.to_numeric(size, errors="coerce")

    # Review rate: Reviews per install
    df_["Review_Rate"] = df_["Reviews"] / (df_["Installs"] + 1)

    # Recency (days since last update)
    reference_date = pd.to_datetime("2018-08-01")
    df_["Last Updated"] = pd.to_datetime(df_["Last Updated"], errors="coerce")
    df_["Recency_Days"] = (reference_date - df_["Last Updated"]).dt.days

    # Price category (categorical feature)
    df_["Price_Category"] = pd.cut(
        df_["Price"].fillna(0),
        bins=[-0.01, 0, 1, 5, 20, np.inf],
        labels=["free", "cheap", "low", "medium", "high"]
    )

    return df_

df = clean_and_engineer_features(df)

### 5. Veri Setini Hazırlama ve Bölme:
Hedef değişken (Success) hariç tutularak özellik matrisi X ve hedef vektörü y belirleniyor. Ardından, veriler eğitim (X_train, y_train) ve test (X_test, y_test) kümelerine bölünüyor. Bu bölme işlemi, hedef değişkenin dengesizliği nedeniyle stratified (katmanlı) olarak yapılıyor.

In [ ]:

drop_cols = ["App", "Rating", "Success", "Current Ver", "Android Ver", "Last Updated"]
drop_cols = [c for c in drop_cols if c in df.columns]

X = df.drop(columns=drop_cols)
y = df["Success"]


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)



Train shape: (6557, 13)
Test shape: (1640, 13)


### 6. Ön İşleme Hattı (Preprocessing Pipeline) Oluşturma:
- Model eğitimi öncesinde gerekli veri temizleme (imputation), ölçekleme (scaling) ve kodlama (encoding) adımlarını otomatik olarak uygulayacak bir ColumnTransformer oluşturuluyor.

- Sayısal Özellikler için İşlemler:

SimpleImputer(strategy="median"): Eksik değerleri medyan ile doldur.

StandardScaler(): Veriyi standartlaştır (ortalama 0, standart sapma 1).

Kategorik Özellikler için İşlemler:

SimpleImputer(strategy="most_frequent"): Eksik değerleri en sık görülen değer ile doldur.

OneHotEncoder(handle_unknown="ignore"): Kategorik değerleri One-Hot Encoding ile sayısal hale getir.

In [ ]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

print("\nNumeric features:", numeric_features)
print("Categorical features:", categorical_features)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)


Numeric features: ['Reviews', 'Size', 'Installs', 'Price', 'Avg_Sentiment_Polarity', 'Avg_Sentiment_Subjectivity', 'Review_Rate', 'Recency_Days']
Categorical features: ['Category', 'Type', 'Content Rating', 'Genres', 'Price_Category']


### 7. Özellik Seçimi (Feature Selection):
- HazırlığıSelectKBest metodu, istatistiksel testlere dayanarak en iyi K tane özelliği seçmek için kullanılacak. f_classif (ANOVA F-değeri), sınıflandırma problemlerinde bağımlı değişkenle en alakalı özellikleri bulmak için kullanılır. Burada K=75 olarak belirleniyor.

In [ ]:
K = 75

anova_selector = SelectKBest(score_func=f_classif, k=K)

### 8. Modeller ve Değerlendirme Fonksiyonu:
Bu blokta kullanılacak üç doğrusal model (LogisticRegression, SGDClassifier, LinearSVC) tanımlanıyor ve modelleri eğiten, tahmin yapan ve performans metriklerini hesaplayan bir fonksiyon oluşturuluyor.

Model Seçimi Notu: Sınıf dengesizliğini ele almak için LogisticRegression ve LinearSVC'ye varsayılan olarak class_weight="balanced" parametresi eklenmiş (SGDClassifier için sonraki adımlarda bu parametre ayarlanacaktır).

evaluate_model Fonksiyonu Notu: roc_auc_score'u hesaplamak için modelin olasılık (predict_proba) veya karar fonksiyonu (decision_function) çıktısını kullanıp kullanamadığı kontrol ediliyor. LinearSVC gibi bazı modeller olasılık yerine karar skorları verir. SGDClassifier'da loss="log_loss" kullanıldığında, aslında L2 düzenlemesi olan bir Lojistik Regresyon eğitilir ve bu da olasılık çıktısı verir.

In [ ]:
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42
    ),
    "SGDClassifier": SGDClassifier(
        loss="log_loss", max_iter=2000, alpha=1e-4, random_state=42
    ),
    "LinearSVC": LinearSVC(
        random_state=42
    ),
}

def make_baseline_pipeline(clf):
    """Pipeline بدون Feature Selection (فقط preprocessing + model)."""
    return Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("clf", clf),
        ]
    )

def make_anova_pipeline(clf):
    """Pipeline مع SelectKBest(ANOVA)."""
    return Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("select", anova_selector),
            ("clf", clf),
        ]
    )

def evaluate_model(pipeline, X_tr, X_te, y_tr, y_te, model_name, variant):
    pipeline.fit(X_tr, y_tr)
    y_pred = pipeline.predict(X_te)


    if hasattr(pipeline, "predict_proba"):
        y_score = pipeline.predict_proba(X_te)[:, 1]
    elif hasattr(pipeline, "decision_function"):
        y_score = pipeline.decision_function(X_te)
    else:
        y_score = None

    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred)
    rec = recall_score(y_te, y_pred)
    f1 = f1_score(y_te, y_pred)
    if y_score is not None:
        roc = roc_auc_score(y_te, y_score)
    else:
        roc = np.nan

    print(f"\n=== {model_name} ({variant}) ===")
    print("Accuracy :", acc)
    print("Precision:", prec)
    print("Recall   :", rec)
    print("F1       :", f1)
    if not np.isnan(roc):
        print("ROC AUC  :", roc)
    print("\nClassification report:")
    print(classification_report(y_te, y_pred))

    return {
        "Model": model_name,
        "Variant": variant,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "ROC_AUC": roc,
    }

results = []

for name, clf in models.items():

    baseline_pipe = make_baseline_pipeline(clf)
    res_base = evaluate_model(
        baseline_pipe, X_train, X_test, y_train, y_test, name, "No_FS"
    )
    results.append(res_base)


    anova_pipe = make_anova_pipeline(clf)
    res_anova = evaluate_model(
        anova_pipe, X_train, X_test, y_train, y_test, name, f"ANOVA_k={K}"
    )
    results.append(res_anova)

E:\downloads\Anaconda\Lib\site-packages\sklearn\impute\_base.py:598: UserWarning: Skipping features without any observed values: ['Recency_Days']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
E:\downloads\Anaconda\Lib\site-packages\sklearn\impute\_base.py:598: UserWarning: Skipping features without any observed values: ['Recency_Days']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
E:\downloads\Anaconda\Lib\site-packages\sklearn\impute\_base.py:598: UserWarning: Skipping features without any observed values: ['Recency_Days']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
E:\downloads\Anaconda\Lib\site-packages\sklearn\impute\_base.py:598: UserWarning: Skipping features without any observed values: ['Recency_Days']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
E:\downloads\Anaconda\Li


=== LogisticRegression (No_FS) ===
Accuracy : 0.6060975609756097
Precision: 0.8933161953727506
Recall   : 0.5524642289348172
F1       : 0.6827111984282908
ROC AUC  : 0.7211188706415068

Classification report:
              precision    recall  f1-score   support

           0       0.35      0.78      0.48       382
           1       0.89      0.55      0.68      1258

    accuracy                           0.61      1640
   macro avg       0.62      0.67      0.58      1640
weighted avg       0.77      0.61      0.64      1640


=== LogisticRegression (ANOVA_k=75) ===
Accuracy : 0.6109756097560975
Precision: 0.900516795865633
Recall   : 0.5540540540540541
F1       : 0.6860236220472441
ROC AUC  : 0.7161308983760477

Classification report:
              precision    recall  f1-score   support

           0       0.35      0.80      0.49       382
           1       0.90      0.55      0.69      1258

    accuracy                           0.61      1640
   macro avg       0.63      0.

E:\downloads\Anaconda\Lib\site-packages\sklearn\impute\_base.py:598: UserWarning: Skipping features without any observed values: ['Recency_Days']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
E:\downloads\Anaconda\Lib\site-packages\sklearn\impute\_base.py:598: UserWarning: Skipping features without any observed values: ['Recency_Days']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
E:\downloads\Anaconda\Lib\site-packages\sklearn\impute\_base.py:598: UserWarning: Skipping features without any observed values: ['Recency_Days']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
E:\downloads\Anaconda\Lib\site-packages\sklearn\impute\_base.py:598: UserWarning: Skipping features without any observed values: ['Recency_Days']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
E:\downloads\Anaconda\Li


=== SGDClassifier (No_FS) ===
Accuracy : 0.7591463414634146
Precision: 0.769182782283219
Recall   : 0.980127186009539
F1       : 0.8619363858790633
ROC AUC  : 0.72354314585605

Classification report:
              precision    recall  f1-score   support

           0       0.32      0.03      0.06       382
           1       0.77      0.98      0.86      1258

    accuracy                           0.76      1640
   macro avg       0.55      0.51      0.46      1640
weighted avg       0.67      0.76      0.67      1640


=== SGDClassifier (ANOVA_k=75) ===
Accuracy : 0.7591463414634146
Precision: 0.769182782283219
Recall   : 0.980127186009539
F1       : 0.8619363858790633
ROC AUC  : 0.7187445375773063

Classification report:
              precision    recall  f1-score   support

           0       0.32      0.03      0.06       382
           1       0.77      0.98      0.86      1258

    accuracy                           0.76      1640
   macro avg       0.55      0.51      0.46   

E:\downloads\Anaconda\Lib\site-packages\sklearn\impute\_base.py:598: UserWarning: Skipping features without any observed values: ['Recency_Days']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
E:\downloads\Anaconda\Lib\site-packages\sklearn\impute\_base.py:598: UserWarning: Skipping features without any observed values: ['Recency_Days']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
E:\downloads\Anaconda\Lib\site-packages\sklearn\impute\_base.py:598: UserWarning: Skipping features without any observed values: ['Recency_Days']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


9. Sonuçları Özetleme
Tüm model varyasyonlarının performans metrikleri bir DataFrame'de toplanıyor ve F1 skoruna göre büyükten küçüğe sıralanarak en iyi performans gösteren model belirleniyor.

In [ ]:
results_df = pd.DataFrame(results)
print("\n\n===== Summary Results  =====")
print(results_df.sort_values(by="F1", ascending=False))



===== Summary Results  =====
                Model     Variant  Accuracy  Precision    Recall        F1  \
5           LinearSVC  ANOVA_k=75  0.761585   0.767098  0.989666  0.864283   
4           LinearSVC       No_FS  0.761585   0.767428  0.988871  0.864189   
2       SGDClassifier       No_FS  0.759146   0.769183  0.980127  0.861936   
3       SGDClassifier  ANOVA_k=75  0.759146   0.769183  0.980127  0.861936   
1  LogisticRegression  ANOVA_k=75  0.610976   0.900517  0.554054  0.686024   
0  LogisticRegression       No_FS  0.606098   0.893316  0.552464  0.682711   

    ROC_AUC  
5  0.705271  
4  0.712007  
2  0.723543  
3  0.718745  
1  0.716131  
0  0.721119  


Çıktı Yorumu: Özellik seçimi (ANOVA) olmayan LinearSVC modeli en yüksek F1 skorunu (0.8689) elde etmiş, ancak ROC AUC skoru en yüksek olan LogisticRegression (No_FS) (0.7306) olmuştur.